# DINOv2 ViT-S/14 — DIMER image feature extraction tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/dinov2-feature-extraction-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/dinov2-feature-extraction-pipeline/blob/main/tutorials/dinov2_feature_extraction_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-timm%2Fvit__small__patch14__dinov2.lvd142m-ffcc4d?style=flat)](https://huggingface.co/timm/vit_small_patch14_dinov2.lvd142m) [![Upstream](https://img.shields.io/badge/Upstream-facebookresearch%2Fdinov2-181717?style=flat&logo=github&logoColor=white)](https://github.com/facebookresearch/dinov2) [![arXiv](https://img.shields.io/badge/arXiv-2304.07193-b31b1b.svg)](https://arxiv.org/abs/2304.07193)

**Profile:** `TASK-INFERENCE`  
**Notebook specification:** DIMER Notebook Specification 1.1 — **standalone** (§3.6)  
**Capability:** self-supervised image feature extraction (one 384-d embedding per image) using the pinned DINOv2 ViT-S/14 `lvd142m` weights

**This notebook is standalone.** It carries the repository's pipeline module (`src/dinov2_feature_extraction_pipeline/pipeline.py` at revision `2d8c124ed0b2`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `4610ca143709d58a633b6397a74412c2c3842454` (~88 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

At inference each image is resized so its shorter side is 518 px, centre-cropped to 518 x 518, normalised with the ImageNet mean/std from the snapshot config, and passed through the ViT-S/14 backbone with no classifier head; the class token after the final LayerNorm is taken as the image's feature (`POOLING = "cls"`) and L2-normalised (`NORMALIZED = True`), so a dot product between two vectors is their cosine similarity. **Embeddings are representations, not predictions:** the pipeline returns no label, no score, no class and no threshold, and no intrinsic accuracy exists for a vector on its own. **No adaptation occurs:** no training, fine-tuning, in-context conditioning, or preprocessing fitting happens in this notebook — the upstream checkpoint supplies the architecture, weights and preprocessing configuration, and the carried pipeline module adds snapshot verification, input validation, the class-token pooling choice, L2 normalisation, a fixed output contract and the `validate_inputs` and `evaluation_report` helpers.

**Learning objectives:** install the pinned runtime, read what the carried pipeline module guarantees, resolve and digest-verify the immutable upstream model revision, generate a small synthetic input set and validate it into an input manifest, embed a batch through the public API, read the embedding output (shape, unit, pooling, normalisation) correctly, run a qualitative cosine-similarity check and understand why it is not a metric, produce an evaluation report that is honestly `not-measurable` and says what downstream task would make the features measurable, and export the vectors with their identifiers plus provenance.

**This notebook does not demonstrate:** image classification, object detection, segmentation, captioning, image-text comparison, patch-level (dense) features, attention maps, or any similarity search, clustering or probe — the carried module exposes only the pooled per-image vector; everything downstream is the caller's code.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU and uses CUDA automatically when available; inference is float32 on both. Each 518 px image costs about 46.8 GMACs (upstream card), so a three-image CPU batch takes seconds to tens of seconds on a hosted CPU runtime; on the model card's GPU (RTX 5070 Ti) the verified snapshot loaded in 3.4 s and a two-image batch took 1.2 s. The pinned `torch==2.14.0` install and the 88 MB checkpoint are the largest downloads of the run.
- **Knowledge:** basic Python and NumPy; what an embedding vector is and why cosine similarity between two vectors is not an accuracy.
- **Data:** the default sample is a set of three synthetic images generated in code, so nothing is downloaded and no private data is needed. Optional BYOD upload is gated off by default so the sample path can run top-to-bottom without interaction. Expected BYOD input: one or more image files decodable by Pillow (PNG/JPEG/WebP and similar), any colour mode, each side at most 4096 px, at most `MAX_BATCH` files per run. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **External access:** the Hugging Face Hub only, to fetch the pinned `timm/vit_small_patch14_dinov2.lvd142m` snapshot (~88 MB in total) at revision `4610ca143709…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `timm` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'timm==1.0.29',
    'huggingface-hub==0.36.2',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
]
NOTEBOOK_SOURCE = {
    'repository': 'dinov2-feature-extraction-pipeline',
    'repository_revision': '2d8c124ed0b2ac4007505c3b65c87d709fb9f1ea',
    'embedded_module': 'src/dinov2_feature_extraction_pipeline/pipeline.py',
    'embedded_modules': ['src/dinov2_feature_extraction_pipeline/pipeline.py'],
    'module_sha256': '98d713de690a297d2b5378c8065004a6e858652adfaf60f09bf80690388450b9',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '1.1',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, timm
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'timm': timm.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/dinov2_feature_extraction_pipeline/` @ `2d8c124ed0b2`)

The next 1 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/1:** `src/dinov2_feature_extraction_pipeline/pipeline.py`

In [ ]:
"""Self-supervised image feature extraction with the pinned ``timm/vit_small_patch14_dinov2.lvd142m`` weights.

DINOv2 ViT-S/14 with no classifier head (``num_classes=0``): each image becomes one 384-d vector,
the class token after the final norm (``POOLING = "cls"``), L2-normalised here. Weights load only
from a digest-verified local snapshot (``weights/<key>/``) or, when explicitly allowed, from the
Hugging Face Hub at the pinned revision. Preprocessing is the upstream ``pretrained_cfg``.
"""

from __future__ import annotations

import hashlib
import json
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

from PIL import Image

MODEL_ID = "timm/vit_small_patch14_dinov2.lvd142m"
MODEL_REVISION = "4610ca143709d58a633b6397a74412c2c3842454"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "vit-small-dinov2"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"
WEIGHTS_FILE = "model.safetensors"
CONFIG_FILE = "config.json"

EMBED_DIM = 384  # ViT-S width; the snapshot config.json reports num_features = 384
POOLING = "cls"  # snapshot config.json global_pool = "token": the class token after the final LayerNorm
NORMALIZED = True  # embed() L2-normalises every vector, so a dot product is a cosine similarity
MAX_IMAGE_SIDE = 4096  # pixels; larger images are rejected before any decode-to-tensor work
MAX_BATCH = 32  # images per embed() call; 518-px inputs cost 46.8 GMACs each (upstream card)


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its manifest; raise naming the first mismatch."""
    root = Path(path or DEFAULT_WEIGHTS_DIR)
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest.get("files", []):
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {"path": str(root), **manifest}


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def _hub_reference(model_id: str, revision: str) -> str:
    """timm's ``hf-hub:owner/name@revision`` form; ``hf_split`` passes ``revision=`` to hf_hub_download."""
    return f"hf-hub:{model_id}@{revision}"


INPUT_SCHEMA: dict[str, Any] = {
    "input": "PIL.Image.Image or a sequence of them; any mode, converted to RGB",
    "image_side_px": [1, MAX_IMAGE_SIDE],
    "batch": [1, MAX_BATCH],
    "output": f"one L2-normalised {EMBED_DIM}-d vector per image, pooling {POOLING!r}",
    "preprocessing": (
        "resize shorter side to 518 px, center-crop 518x518 (crop_pct 1.0, bicubic), ImageNet mean/std"
    ),
}


def _check_inputs(images: Any) -> list[Image.Image]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the images as a list."""
    if isinstance(images, Image.Image):
        images = [images]
    if not isinstance(images, Sequence) or isinstance(images, str | bytes):
        raise TypeError("images must be a PIL.Image.Image or a sequence of them")
    if not 1 <= len(images) <= MAX_BATCH:
        raise ValueError(f"batch size must be between 1 and MAX_BATCH={MAX_BATCH}, got {len(images)}")
    for image in images:
        if not isinstance(image, Image.Image):
            raise TypeError(f"each image must be a PIL.Image.Image, got {type(image).__name__}")
        width, height = image.size
        if width < 1 or height < 1 or max(width, height) > MAX_IMAGE_SIDE:
            raise ValueError(f"image side outside 1..MAX_IMAGE_SIDE={MAX_IMAGE_SIDE} px: {image.size}")
    return list(images)


def validate_inputs(
    images: Image.Image | Sequence[Image.Image], *, names: Sequence[str] | None = None
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, per-input observations, verdict).

    Rejection is reported by raising exactly as ``embed`` would; a caller that wants the finding
    recorded catches the exception and stores ``str(exc)`` under ``findings``.
    """
    checked = _check_inputs(images)
    if names is not None and len(names) != len(checked):
        raise ValueError("names must have one entry per image")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [
            {"id": names[i] if names else f"image-{i}", "mode": image.mode, "size": list(image.size)}
            for i, image in enumerate(checked)
        ],
        "batch_size": len(checked),
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    result: Mapping[str, Any], targets: Sequence[Any] | None = None, *, sample_kind: str = "synthetic"
) -> dict[str, Any]:
    """Evaluation stage: always ``not-measurable`` — the output is a representation (EVAL9).

    The repository ships no metric helper because a vector has nothing to be scored against on its
    own. ``targets`` is accepted so the signature matches the fleet's other pipelines, but class or
    relevance labels alone cannot score an embedding: the report stays ``not-measurable`` and names
    the downstream task that would make the features measurable.
    """
    embeddings = result["embeddings"]
    reason = "an embedding is a representation, not a prediction: no intrinsic performance measure exists"
    if targets is not None:
        reason = (
            "targets were supplied, but this pipeline exposes no metric helper and labels alone cannot "
            "score a representation; score them through a downstream task instead"
        )
    return {
        "task": "self-supervised image feature extraction (pooled per-image embedding)",
        "score_semantics": (
            f"cosine similarity between two L2-normalised {EMBED_DIM}-d vectors (a dot product); "
            "a similarity is not a probability, an accuracy or a calibrated score; no threshold is shipped"
        ),
        "sample_kind": sample_kind,
        "n_embeddings": len(embeddings),
        "metrics": [],
        "baselines": [],
        "verdict": "not-measurable",
        "reason": reason,
        "needs": (
            "a downstream labelled task: a retrieval set with relevance labels (mean average precision), "
            "a labelled image set for a linear probe or k-NN classifier (accuracy), or human-judged "
            "duplicate pairs to calibrate a similarity threshold"
        ),
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


@dataclass
class DINOv2FeatureExtractionPipeline:
    """``_runner`` maps a float tensor (N, 3, H, W) to pooled features (N, EMBED_DIM); injectable."""

    _runner: Callable[[Any], Any]
    _transform: Callable[[Image.Image], Any]
    device: str = "cpu"
    input_size: tuple[int, int] = (0, 0)
    source: str = "injected"

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> DINOv2FeatureExtractionPipeline:
        root = Path(weights_dir or DEFAULT_WEIGHTS_DIR)
        arch_name = MODEL_ID.split("/", 1)[1]
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            with open(root / CONFIG_FILE, encoding="utf-8") as fh:
                config = json.load(fh)
            snapshot_name = f"{config['architecture']}.{config['pretrained_cfg']['tag']}"
            if snapshot_name != arch_name:
                raise ValueError(f"snapshot config names {snapshot_name!r}, expected {arch_name!r}")
            overlay = dict(config["pretrained_cfg"])
            overlay["file"] = str(root / WEIGHTS_FILE)  # 'file' takes precedence over hf_hub_id in timm
            import timm  # after snapshot verification / download consent

            model = timm.create_model(
                arch_name, pretrained=True, pretrained_cfg_overlay=overlay, num_classes=0
            )
            source = "local-snapshot"
        elif allow_download:
            import timm  # after snapshot verification / download consent

            model = timm.create_model(
                _hub_reference(MODEL_ID, revision=MODEL_REVISION), pretrained=True, num_classes=0
            )
            source = "hf-hub"
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage it with: hf download {MODEL_ID} --revision {MODEL_REVISION} --local-dir {root}"
            )
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from timm.data import create_transform, resolve_model_data_config

        if getattr(model, "num_features", EMBED_DIM) != EMBED_DIM:
            raise ValueError(f"model num_features {model.num_features} != EMBED_DIM={EMBED_DIM}")
        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        model = model.eval().to(resolved_device)
        data_config = resolve_model_data_config(model)
        transform = create_transform(**data_config, is_training=False)
        input_size = tuple(int(s) for s in data_config["input_size"][1:])

        def runner(batch: Any) -> Any:
            with torch.inference_mode():
                return model(batch.to(resolved_device))  # num_classes=0 -> pooled class token, (N, 384)

        return cls(runner, transform, resolved_device, input_size, source)

    def _validate(self, images: Any) -> list[Image.Image]:
        return _check_inputs(images)

    def embed(self, images: Image.Image | Sequence[Image.Image]) -> dict[str, Any]:
        """Return one L2-normalised float32 vector of length EMBED_DIM per image; no labels, no scores."""
        import torch

        batch_images = self._validate(images)
        batch = torch.stack([self._transform(image.convert("RGB")) for image in batch_images])
        features = self._runner(batch)
        if not isinstance(features, torch.Tensor) or features.shape != (len(batch_images), EMBED_DIM):
            raise RuntimeError("runner must return a tensor of shape (batch, EMBED_DIM)")
        vectors = torch.nn.functional.normalize(features.float(), dim=-1).cpu()
        return {
            "embeddings": [[float(v) for v in row] for row in vectors.tolist()],
            "dim": EMBED_DIM,
            "pooling": POOLING,
            "normalized": NORMALIZED,
            "input_size": list(self.input_size),
            "device": self.device,
            "source": self.source,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `3`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `4610ca143709…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `DINOv2FeatureExtractionPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "vit-small-dinov2",
  "modelId": "timm/vit_small_patch14_dinov2.lvd142m",
  "revision": "4610ca143709d58a633b6397a74412c2c3842454",
  "files": [
    {
      "path": "README.md",
      "bytes": 4007,
      "sha256": "435b14d32b87c5e933b7a5f4ca187abc5a682a41a4ee28aa2ac16d949a53f86f"
    },
    {
      "path": "config.json",
      "bytes": 615,
      "sha256": "b651fc1b08edf7d1c15a121cbb3270c802bd87901cfd4860d7ecb286e7d52a05"
    },
    {
      "path": "model.safetensors",
      "bytes": 88240510,
      "sha256": "04d27f3400d059fc0cfd7d17dd1909a75bf3ea8fb3eeb48b97cb99e57ee20081"
    }
  ],
  "totalBytes": 88245132
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = DINOv2FeatureExtractionPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Generate the synthetic sample set or optional BYOD

The default sample is **synthetic**: three 256 x 256 RGB images built in code — a deterministic gradient (red ramps left to right, green top to bottom, blue is their mean), the same gradient rotated by 180 degrees, and a flat mid-grey block. They need no download, contain no personal data, and each one's pixel SHA-256 is printed and exported alongside its identifier; no randomness is involved, so no seed is needed. None of them is a photograph, so the embeddings they produce are sanity evidence that the code path works: the rotated copy is there only so that Section 7 can show a *qualitative* similarity comparison (a near-duplicate versus an unrelated image), which is not a benchmark and not a metric. BYOD is optional and disabled by default; when enabled, upload one or more image files and they replace the synthetic set.

In [ ]:
import hashlib
import io

import numpy as np
from PIL import Image

USE_BYOD = False  # @param {type:"boolean"}
SAMPLE_SIDE = 256

if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    images = {}
    for name, data in uploaded.items():
        image = Image.open(io.BytesIO(data))
        image.load()
        images[name] = image
    sample_kind = 'BYOD upload'
else:
    # Deterministic synthetic set: no randomness, so no seed is needed and the digests are stable.
    ramp = np.linspace(0.0, 255.0, SAMPLE_SIDE)
    red = np.tile(ramp, (SAMPLE_SIDE, 1))
    green = red.T
    blue = (red + green) / 2.0
    gradient = Image.fromarray(np.rint(np.stack([red, green, blue], axis=-1)).astype(np.uint8), mode='RGB')
    images = {
        f'synthetic_gradient_{SAMPLE_SIDE}': gradient,
        f'synthetic_gradient_{SAMPLE_SIDE}_rot180': gradient.rotate(180),
        f'synthetic_flat_grey_{SAMPLE_SIDE}': Image.new('RGB', (SAMPLE_SIDE, SAMPLE_SIDE), (128, 128, 128)),
    }
    sample_kind = 'synthetic'
image_ids = list(images)
digests = {name: hashlib.sha256(np.asarray(image.convert('RGB')).tobytes()).hexdigest() for name, image in images.items()}
for name in image_ids:
    print({'id': name, 'mode': images[name].mode, 'size': images[name].size, 'pixel_sha256': digests[name]})
print({'sample_kind': sample_kind, 'count': len(image_ids)})

## 5. Validate the inputs → input manifest

`validate_inputs` is the pipeline's public validation stage: it applies exactly the checks `embed` applies — type, batch size 1..`MAX_BATCH`, image side 1..`MAX_IMAGE_SIDE` px — and returns an **input manifest** naming the schema and ceilings, each input's identifier, observed mode and size, and the verdict. The manifest is written to `outputs/dinov2_feature_extraction_input_manifest.json`. The cell first prints the ceilings and the embedding contract — `EMBED_DIM` (vector length, 384), `POOLING` (`"cls"`: the class token after the final LayerNorm — one vector **per image**, not per patch or per token) and `NORMALIZED` (`True`: every vector has unit L2 norm) — so the values shown are the ones in force before any model work. To show what rejection looks like, it also validates a deliberately oversized image and records the pipeline's own error message as a finding. **What the pipeline changes about your images:** each is converted to RGB, resized so its shorter side is 518 px and centre-cropped to 518 x 518 (`crop_pct: 1.0`, `crop_mode: "center"` in the snapshot config) — for a non-square image the outer strips along the longer side never reach the encoder; nothing else is dropped, and there is no missing-data concept: every pixel of the cropped square is visible to the encoder. The notebook itself does not resize, crop, or subsample.

In [ ]:
import json
import os

os.makedirs('outputs', exist_ok=True)
print({'ceilings': {'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MAX_BATCH': MAX_BATCH}, 'contract': {'EMBED_DIM': EMBED_DIM, 'POOLING': POOLING, 'NORMALIZED': NORMALIZED}})
input_manifest = validate_inputs([images[name] for name in image_ids], names=image_ids)
# Demonstrate rejection on an input that breaks a ceiling; the finding is recorded, not swallowed.
try:
    validate_inputs(Image.new('RGB', (MAX_IMAGE_SIDE + 1, 8)))
except ValueError as exc:
    input_manifest['findings'].append({'input': 'oversized-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/dinov2_feature_extraction_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print(json.dumps(input_manifest, indent=2))

## 6. Embed the batch

`embed` returns `embeddings` (a list with one 384-float list per input, **in input order**, so position *i* belongs to `image_ids[i]`), plus `dim`, `pooling`, `normalized`, `input_size`, `device`, `source`, `model_id` and `model_revision`. The checks below are falsifiable plumbing checks (one vector per input, length `EMBED_DIM`, finite, unit norm within float tolerance) and the cell raises if any fails; they are not a quality measure. Small numeric differences between CPU and CUDA kernels move each vector slightly, so values are repeatable on fixed hardware but not bitwise-identical across devices. The timing is measured for this batch on the runtime identified in Section 1 and includes the first-call warm-up.

In [ ]:
import time

started = time.perf_counter()
result = pipe.embed([images[name] for name in image_ids])
elapsed = time.perf_counter() - started
vectors = np.asarray(result['embeddings'], dtype=np.float32)
norms = np.linalg.norm(vectors, axis=1)
checks = {
    'one_vector_per_input': vectors.shape[0] == len(image_ids),
    'dim_matches_contract': vectors.shape[1] == result['dim'] == EMBED_DIM,
    'all_finite': bool(np.isfinite(vectors).all()),
    'unit_norm': bool(np.allclose(norms, 1.0, atol=1e-4)),
}
if not all(checks.values()):
    raise RuntimeError(f'embedding output failed a sanity check: {checks}')
print({key: value for key, value in result.items() if key != 'embeddings'})
print({'embeddings_shape': list(vectors.shape), 'norms': [round(float(n), 6) for n in norms], 'seconds': round(elapsed, 3), 'checks': checks})

## 7. Evaluate → evaluation report

The repository ships **no metric helper and reports no performance measure**, because an embedding is a representation, not a prediction — there is nothing to score it against on its own. `evaluation_report` still produces a report, and its verdict is always `not-measurable`: it names the score semantics (a cosine similarity is not a probability, an accuracy, or a calibrated score) and states what would make the features measurable. Evaluating these features requires a downstream labelled task: a retrieval set with relevance labels (mean average precision), a labelled image set for a linear probe or k-NN classifier (accuracy), or human-judged duplicate pairs to calibrate a similarity threshold; the DINOv2 paper evaluates exactly such tasks, and those upstream numbers are not measured here. The pairwise cosine similarities printed below (a dot product, because the vectors are unit-normalised) are a **qualitative check only**: the rotated copy is expected to score higher against the original than the flat block does, which shows that the vector encodes image content, but the absolute values mean nothing without a threshold calibrated on your own labelled pairs, and the pipeline deliberately ships none. The report is written to `outputs/dinov2_feature_extraction_evaluation_report.json`.

In [ ]:
report = evaluation_report(result, sample_kind=sample_kind)
with open('outputs/dinov2_feature_extraction_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
print(json.dumps(report, indent=2))
cosine = vectors @ vectors.T
print('pairwise cosine similarity (qualitative check only; no threshold is shipped):')
for i, name in enumerate(image_ids):
    print(f"{name:>36}  " + '  '.join(f'{cosine[i, j]:.4f}' for j in range(len(image_ids))))

## 8. Export vectors, identifiers, and provenance

One JSON record is written under `outputs/`: an `items` list with, per image, its identifier, pixel digest, size and 384-float vector (so every vector maps back to its input), the embedding contract (`dim`, `pooling`, `normalized`, `input_size`), the pairwise cosine matrix keyed by the same identifiers, the sanity checks, the input manifest, the evaluation report, the notebook's source (repository, revision, embedded module digest, generator), the model identifier, the immutable model revision, the model licence, and the runtime identity (Python, `torch`, `timm`, device). No credentials are involved in any step, so none can reach the export.

In [ ]:
payload = {
    'items': [
        {'id': name, 'pixel_sha256': digests[name], 'width': images[name].width, 'height': images[name].height, 'vector': result['embeddings'][index]}
        for index, name in enumerate(image_ids)
    ],
    'dim': result['dim'],
    'pooling': result['pooling'],
    'normalized': result['normalized'],
    'input_size': result['input_size'],
    'cosine_similarity': {'ids': image_ids, 'matrix': [[round(float(value), 6) for value in row] for row in cosine]},
    'sanity_checks': checks,
    'input_manifest': input_manifest,
    'evaluation_report': report,
    'sample_kind': sample_kind,
    'seconds': round(elapsed, 3),
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'timm': timm.__version__,
        'device': pipe.device,
    },
}
with open('outputs/dinov2_feature_extraction_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

Each output is one L2-normalised 384-float vector per image — a representation, not a prediction. There is no label, no class, no score and no threshold, and the evaluation report says `not-measurable` by construction: a cosine similarity between two vectors is not an accuracy, and the numbers printed on the synthetic set are plumbing evidence only. Every image is resized to shorter side 518 px and centre-cropped to 518 x 518, so for a non-square input the outer strips along the longer side never reach the encoder. The pipeline does not detect out-of-distribution inputs (drawings, scans, satellite tiles), blur, or capture-device drift, and it exposes no patch-level features, no attention maps, and no downstream search, clustering or probe.

Successful execution proves that the recorded repository revision's pipeline module, carried in this notebook, can acquire and digest-verify the pinned model, validate the demonstrated inputs against the enforced ceilings, execute the public pipeline path, and emit the shown machine-readable outputs in the tested runtime — without the repository being reachable. It does **not** establish benchmark superiority, reproduction of the upstream DINOv2 evaluations, retrieval or probe quality on your data, safety for high-consequence decisions, or production fitness on an unseen domain.

**Next experiments:** enable `USE_BYOD` with a handful of your own images, including a near-duplicate pair, and look at whether the cosine ordering matches your judgement; build a small labelled set and fit a linear probe or k-NN classifier on the exported vectors to get a number that actually means something; compare the CUDA and CPU cosine matrices on the same batch to see the size of kernel-level variability.

## References

- Repository README: https://github.com/kurtvalcorza/dinov2-feature-extraction-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/dinov2-feature-extraction-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/dinov2-feature-extraction-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/timm/vit_small_patch14_dinov2.lvd142m
- Upstream code: https://github.com/facebookresearch/dinov2
- DINOv2 paper: https://arxiv.org/abs/2304.07193
- timm documentation: https://huggingface.co/docs/timm